In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DoubleType, FloatType
from pyspark.sql.functions import col, count, length,countDistinct, min as spark_min, max as spark_max,avg, stddev, approx_count_distinct

spark = (SparkSession.builder
                     .appName("MusicBrainz")
                     .config("spark.driver.memory", "2g")
                     .config("spark.executor.memory", "18g")
                     .config('spark.executor.instances', 7)
                     .getOrCreate())

In [7]:
MBDUMP = "/expanse/lustre/projects/uci157/rwheaton1/mbdump"

In [8]:
def peek(df, n=5):
    return display(df.limit(n).toPandas())

Define Schemas

In [9]:
artist_schema = StructType([
    StructField("id",               IntegerType(),   True),
    StructField("gid",              StringType(),    True),
    StructField("name",             StringType(),    True),
    StructField("sort_name",        StringType(),    True),
    StructField("begin_date_year",  IntegerType(),   True),
    StructField("begin_date_month", IntegerType(),   True),
    StructField("begin_date_day",   IntegerType(),   True),
    StructField("end_date_year",    IntegerType(),   True),
    StructField("end_date_month",   IntegerType(),   True),
    StructField("end_date_day",     IntegerType(),   True),
    StructField("type",             IntegerType(),   True),
    StructField("area",             IntegerType(),   True),
    StructField("gender",           IntegerType(),   True),
    StructField("comment",          StringType(),    True),
    StructField("edits_pending",    IntegerType(),   True),
    StructField("last_updated",     StringType(),    True),
    StructField("ended",            StringType(),    True),
    StructField("begin_area",       IntegerType(),   True),
    StructField("end_area",         IntegerType(),   True),
])
instrument_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("type",          IntegerType(), True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
label_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("sort_name",     StringType(),  True),
    StructField("type",          IntegerType(), True),
    StructField("area",          IntegerType(), True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
    StructField("ended",         StringType(),  True),
])
genre_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
area_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("type",          IntegerType(), True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
tag_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("ref_count",     IntegerType(), True),
])
gender_schema = StructType([
    StructField("id",   IntegerType(), True),
    StructField("name", StringType(),  True),
])
release_group_schema = StructType([
    StructField("id",            IntegerType(), True),
    StructField("gid",           StringType(),  True),
    StructField("name",          StringType(),  True),
    StructField("type",          IntegerType(), True),
    StructField("artist_credit", IntegerType(), True),
    StructField("comment",       StringType(),  True),
    StructField("edits_pending", IntegerType(), True),
    StructField("last_updated",  StringType(),  True),
])
l_artist_label_schema = StructType([
    StructField("id",        IntegerType(), True),
    StructField("link",      IntegerType(), True),
    StructField("entity0",   IntegerType(), True),  # artist
    StructField("entity1",   IntegerType(), True),  # label
])
l_artist_release_group_schema = StructType([
    StructField("id",        IntegerType(), True),
    StructField("link",      IntegerType(), True),
    StructField("entity0",   IntegerType(), True),  # artist
    StructField("entity1",   IntegerType(), True),  # release_group
])
label_tag_schema = StructType([
    StructField("label", IntegerType(), True),
    StructField("tag",   IntegerType(), True),
    StructField("count", IntegerType(), True),
])
l_artist_genre_schema = StructType([
    StructField("artist", IntegerType(), True),
    StructField("genre",  IntegerType(), True),
    StructField("count",  IntegerType(), True),
])
l_artist_artist_schema = StructType([
    StructField("id",      IntegerType(), True),
    StructField("link",    IntegerType(), True),
    StructField("entity0", IntegerType(), True),
    StructField("entity1", IntegerType(), True),
])
release_group_tag_schema = StructType([
    StructField("release_group", IntegerType(), True),
    StructField("tag",           IntegerType(), True),
    StructField("count",         IntegerType(), True),
])
artist_tag_schema = StructType([
    StructField("artist", IntegerType(), True),
    StructField("tag",    IntegerType(), True),
    StructField("count",  IntegerType(), True),
])
artist_credit_schema = StructType([
    StructField("id",         IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("artist_count", IntegerType(), True),
])
artist_credit_name_schema = StructType([
    StructField("artist_credit", IntegerType(), True),
    StructField("position",      IntegerType(), True),
    StructField("artist",        IntegerType(), True),
    StructField("name",          StringType(),  True),
    StructField("join_phrase",   StringType(),  True),
])
l_artist_instrument_schema = StructType([
    StructField("id",      IntegerType(), True),
    StructField("link",    IntegerType(), True),
    StructField("entity0", IntegerType(), True),  # artist
    StructField("entity1", IntegerType(), True),  # instrument
])


Data Exploration

In [11]:
schemas = {
    "artist": artist_schema,
    "instrument": instrument_schema,
    "label": label_schema,
    "genre": genre_schema,
    "area": area_schema,
    "tag": tag_schema,
    "gender": gender_schema,
    "release_group": release_group_schema,

    "l_artist_label": l_artist_label_schema,
    "l_artist_release_group": l_artist_release_group_schema,
    "label_tag": label_tag_schema,
    "l_artist_genre": l_artist_genre_schema,
    "l_artist_artist": l_artist_artist_schema,
    "release_group_tag": release_group_tag_schema,
    "artist_tag": artist_tag_schema,
    "artist_credit": artist_credit_schema,
    "artist_credit_name": artist_credit_name_schema,
    "l_artist_instrument": l_artist_instrument_schema,
}

dfs = {}
profile_rows = []

for table_name, schema in schemas.items():
    print(f"\n==============================")
    print(f"Loading table: {table_name}")
    print(f"==============================")

    df = (
        spark.read
        .option("sep", "\t")
        .option("nullValue", r"\N")
        .option("header", "false")
        .option("quote", "")
        .option("escape", "")
        .schema(schema)
        .csv(f"{MBDUMP}/{table_name}")
    )

    dfs[table_name] = df
    
    row_count = df.count()
    print(f"Total {table_name} rows: {row_count}")
    duplicate_count = row_count - df.dropDuplicates().count()
    print(f"Duplicate rows in {table_name}: {duplicate_count}")

    print("=== SCHEMA ===")
    df.printSchema()

    print("=== PEEK ===")
    peek(df)


Loading table: artist
Total artist rows: 2853154
=== SCHEMA ===
root
 |-- id: integer (nullable = true)
 |-- gid: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sort_name: string (nullable = true)
 |-- begin_date_year: integer (nullable = true)
 |-- begin_date_month: integer (nullable = true)
 |-- begin_date_day: integer (nullable = true)
 |-- end_date_year: integer (nullable = true)
 |-- end_date_month: integer (nullable = true)
 |-- end_date_day: integer (nullable = true)
 |-- type: integer (nullable = true)
 |-- area: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- comment: string (nullable = true)
 |-- edits_pending: integer (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- ended: string (nullable = true)
 |-- begin_area: integer (nullable = true)
 |-- end_area: integer (nullable = true)

=== PEEK ===


,id,gid,name,sort_name,begin_date_year,begin_date_month,begin_date_day,end_date_year,end_date_month,end_date_day,type,area,gender,comment,edits_pending,last_updated,ended,begin_area,end_area
0,2252039,fadeb38c-833f-40bc-9d8c-a6383b38b1be,Доктор Сатана,Доктор Сатана,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2021-11-23 07:08:52.479537+00,f,NaN,NaN
1,371203,49add228-eac5-4de8-836c-d75cde7369c3,Pete Moutso,"Moutso, Pete",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,None,0,None,f,NaN,NaN
2,3087346,dfdce491-133d-4e9f-9e48-795587e181b0,UNlT,UNlT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2025-09-14 23:53:28.004798+00,f,NaN,NaN
3,2851271,165a49a0-2b3b-4078-a3c1-905afdc07c0a,Babyglock,Babyglock,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,2024-10-19 03:33:55.474151+00,f,NaN,NaN
4,145773,7b4a548e-a01a-49b7-82e7-b49efeb9732c,Aric Leavitt,"Leavitt, Aric",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,0,None,f,NaN,NaN


=== DUPLICATE ROWS ===
Duplicate rows in artist: 0

Loading table: instrument
Total instrument rows: 1056
=== SCHEMA ===
root
 |-- id: integer (nullable = true)
 |-- gid: string (nullable = true)
 |-- name: string (nullable = true)
 |-- type: integer (nullable = true)
 |-- edits_pending: integer (nullable = true)
 |-- last_updated: string (nullable = true)

=== PEEK ===


,id,gid,name,type,edits_pending,last_updated
0,687,c1dbb66d-2356-417a-81ad-f688fee33257,guitarrón mexicano,2,0,2015-02-15 07:32:32.570132+00
1,695,2474c241-d267-433a-a404-688b13c51d11,jouhikko,2,0,2015-02-25 19:47:04.610414+00
2,701,c0cc863c-ea65-4b8a-b365-28b81b72d846,friction idiophone,3,0,2015-02-26 10:28:49.766548+00
3,706,5f9bb15a-738f-48bb-8676-55e62547726f,doshpuluur,2,0,2015-02-28 00:15:24.010482+00
4,707,9bae90ea-9729-4c30-bd3e-f8319cdf4051,igil,2,0,2015-02-28 23:39:39.01131+00


=== DUPLICATE ROWS ===
Duplicate rows in instrument: 0

Loading table: label
Total label rows: 334811
=== SCHEMA ===
root
 |-- id: integer (nullable = true)
 |-- gid: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sort_name: string (nullable = true)
 |-- type: integer (nullable = true)
 |-- area: integer (nullable = true)
 |-- comment: string (nullable = true)
 |-- edits_pending: integer (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- ended: string (nullable = true)

=== PEEK ===


...

=== DUPLICATE ROWS ===
Duplicate rows in artist_credit: 0

Loading table: artist_credit_name
Total artist_credit_name rows: 6801672
=== SCHEMA ===
root
 |-- artist_credit: integer (nullable = true)
 |-- position: integer (nullable = true)
 |-- artist: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- join_phrase: string (nullable = true)

=== PEEK ===


,artist_credit,position,artist,name,join_phrase
0,578352,0,578352,Gustav Ruppke,None
1,273232,0,273232,Zachary,None
2,153193,0,153193,The High Level Ranters,None
3,32262,0,32262,Georges Brassens,None
4,1389968,0,1171184,Harvard of the South,None


=== DUPLICATE ROWS ===
Duplicate rows in artist_credit_name: 0

Loading table: l_artist_instrument
Total l_artist_instrument rows: 96
=== SCHEMA ===
root
 |-- id: integer (nullable = true)
 |-- link: integer (nullable = true)
 |-- entity0: integer (nullable = true)
 |-- entity1: integer (nullable = true)

=== PEEK ===


,id,link,entity0,entity1
0,1,311992,1218252,121
1,2,312066,1395197,19
2,24,393478,1261488,808
3,4,312066,1395235,118
4,5,312066,1395197,21


=== DUPLICATE ROWS ===
Duplicate rows in l_artist_instrument: 0


**Explore and Sumarize Columns**

Attempted a profiling loop that classifies columns by their role and then summarizes them. The MusicBrainz dataset has a lot of IDs and relationship table columns so we need to sift through it and figure out which numeric columns should be summarized like a continuous variable verses an ID. The loop reports nulls, distinct counts, duplicates, categorical frequencies, numeric summaries, and foreign key uniqueness depending on the column type. 

I started by idetifying the primary keys, global ID, forgein key, date part, boolean, text descriptor and count measure clumns for each of the tables we chose. **Please let me know if you disagree with any of the categorizations below.**

I then created the function __get_column_role__ that classifies a given column's role/data type. This function is executed in the for loop which loops through each table's columns to summarize them based on their role which was assigned in the __get_column_role__ function. 

- ID (or id-ish) columns like id, gid, entity0, and entity1, the loop returns missing values, distinct counts, and if foreign keys match the referenced table (not sure if this is actually working how i want it to so take this part with a grain of salt). Not showing frequency distributions bc most ID values are unique and a top 10 count table wouldnt be value add

- Categorical columns are summarized using value counts and percentages to show which values are most common and whether the distribution is balanced or skewed

- Numeric/count columns are summarized with count, mean, standard deviation, min, quartiles, and max

- High-cardinality text columns, like names or UUIDs, get summarized with distinct counts and length stats instead of value counts bc there are way too many unique values

In [17]:
primary_key_cols = {"id"}
global_id_cols = {"gid"}
foreign_key_map = {
    "artist": {
        "type": ("artist_type", "id"),
        "area": ("area", "id"),
        "gender": ("gender", "id"),
        "begin_area": ("area", "id"),
        "end_area": ("area", "id"),
    },
    "instrument": {
        "type": ("instrument_type", "id"),
    },
    "label": {
        "type": ("label_type", "id"),
        "area": ("area", "id"),
    },
    "release_group": {
        "type": ("release_group_primary_type", "id"),
        "artist_credit": ("artist_credit", "id"),
    },

    "artist_credit_name": {
        "artist_credit": ("artist_credit", "id"),
        "artist": ("artist", "id"),
    },

    "label_tag": {
        "label": ("label", "id"),
        "tag": ("tag", "id"),
    },
    "artist_tag": {
        "artist": ("artist", "id"),
        "tag": ("tag", "id"),
    },
    "artist_genre": {
        "artist": ("artist", "id"),
        "genre": ("genre", "id"),
    },
    "release_group_tag": {
        "release_group": ("release_group", "id"),
        "tag": ("tag", "id"),
    },
    "release_group_genre": {
        "release_group": ("release_group", "id"),
        "genre": ("genre", "id"),
    },

    "l_artist_label": {
        "link": ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("label", "id"),
    },
    "l_artist_release_group": {
        "link": ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("release_group", "id"),
    },
    "l_artist_artist": {
        "link": ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("artist", "id"),
    },
    "l_artist_instrument": {
        "link": ("link", "id"),
        "entity0": ("artist", "id"),
        "entity1": ("instrument", "id"),
    },
}
date_part_cols = {"begin_date_year", "begin_date_month", "begin_date_day", "end_date_year", "end_date_month", "end_date_day"}
boolean_like_cols = {"ended"}
text_descriptor_cols = {"name", "sort_name", "comment", "join_phrase", "last_updated"}
count_measure_cols = {"count", "ref_count", "artist_count", "edits_pending", "position"}


In [18]:
def get_column_role(table_name, column_name, row_count, distinct_count):
    distinct_ratio = distinct_count / row_count if row_count else 0

    table_fk = foreign_key_map.get(table_name, {})

    rules = [
                (column_name in primary_key_cols, "primary_key"),
                (column_name in global_id_cols, "global_identifier"),
                (column_name in table_fk, "foreign_key"),
                (column_name in date_part_cols, "date_part"),
                (column_name in boolean_like_cols, "boolean_like"),
                (column_name in count_measure_cols, "count_measure"),
            ]

    for condition, role in rules:
        if condition:
            return role

    if column_name in text_descriptor_cols:
        return "high_cardinality_text" if distinct_ratio > 0.9 else "categorical_text"

    if distinct_ratio > 0.9:
        return "identifier_like"

    if distinct_count <= 25:
        return "categorical"

    return "numeric_or_high_cardinality"

In [22]:
profile_rows = []

for table_name, df in dfs.items():
    print(f"\n\n==============================")
    print(f"TABLE: {table_name}")
    print(f"==============================")

    for field in df.schema.fields:
        column_name = field.name
        dtype = field.dataType.simpleString()

        missing_count = df.filter(col(column_name).isNull()).count()
        distinct_count = df.select(column_name).distinct().count()
        distinct_ratio = distinct_count / row_count if row_count else 0

        role = get_column_role(table_name, column_name, row_count, distinct_count)

        print(f"\n--- {column_name} ({dtype}) ---")
        print(f"Role: {role}")
        print(f"Missing: {missing_count}")
        print(f"Distinct: {distinct_count}")
        print(f"Distinct ratio: {distinct_ratio:.4f}")

        summary = {
            "table": table_name,
            "column": column_name,
            "dtype": dtype,
            "role": role,
            "row_count": row_count,
            "missing_count": missing_count,
            "missing_pct": missing_count / row_count if row_count else None,
            "distinct_count": distinct_count,
            "distinct_ratio": distinct_ratio,
            "duplicate_rows_in_table": duplicate_count,
        }

        if role in ["primary_key", "global_identifier", "identifier_like"]:
            print("Summary: identifier column; frequency distributions are not meaningful.")

            if isinstance(field.dataType, StringType):
                df.select(
                    spark_min(length(col(column_name))).alias("min_length"),
                    spark_max(length(col(column_name))).alias("max_length")
                ).show()

        elif role == "foreign_key":
            ref_table, ref_col = foreign_key_map[table_name][column_name]
            print(f"References: {ref_table}.{ref_col}")

            if ref_table in dfs:
                unmatched_count = (
                    df.select(column_name).where(col(column_name).isNotNull()).distinct()
                    .join(
                        dfs[ref_table].select(col(ref_col).alias("ref_id")).distinct(),
                        col(column_name) == col("ref_id"),
                        "left_anti"
                    )
                    .count()
                )
                print(f"Unmatched foreign keys: {unmatched_count}")
                summary["references"] = f"{ref_table}.{ref_col}"
                summary["unmatched_fk_count"] = unmatched_count
            else:
                print(f"Reference table {ref_table} not loaded.")
                summary["references"] = f"{ref_table}.{ref_col}"
                summary["unmatched_fk_count"] = None

        elif role in ["categorical", "categorical_text", "boolean_like", "date_part"]:
            print("Top values:")
            (
                df.groupBy(column_name)
                .count()
                .withColumn("pct", col("count") / row_count)
                .orderBy(col("count").desc())
                .show(10, truncate=False)
            )

        elif role in ["count_measure", "numeric_or_high_cardinality"]:
            print("Numeric summary:")
            df.select(column_name).summary(
                "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
            ).show()

        elif role == "high_cardinality_text":
            print("Summary: high-cardinality text; showing length stats instead of value counts.")
            df.select(
                spark_min(length(col(column_name))).alias("min_length"),
                spark_max(length(col(column_name))).alias("max_length"),
                avg(length(col(column_name))).alias("avg_length")
            ).show()

        profile_rows.append(summary)

profile_df = spark.createDataFrame(profile_rows)
profile_df.show(200, truncate=False)



TABLE: artist

--- id (int) ---
Role: primary_key
Missing: 0
Distinct: 2853154
Distinct ratio: 1.0000
Summary: identifier column; frequency distributions are not meaningful.

--- gid (string) ---
Role: global_identifier
Missing: 0
Distinct: 2853154
Distinct ratio: 1.0000
Summary: identifier column; frequency distributions are not meaningful.
+----------+----------+
|min_length|max_length|
+----------+----------+
|        36|        36|
+----------+----------+


--- name (string) ---
Role: high_cardinality_text
Missing: 0
Distinct: 2569313
Distinct ratio: 0.9005
Summary: high-cardinality text; showing length stats instead of value counts.
+----------+----------+------------------+
|min_length|max_length|        avg_length|
+----------+----------+------------------+
|         1|      1018|12.070556303655534|
+----------+----------+------------------+


--- sort_name (string) ---
Role: categorical_text
Missing: 0
Distinct: 2564240
Distinct ratio: 0.8987
Top values:
+---------+-----+----

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: An error occurred while calling o3224.count